In [63]:
import rdflib
import pandas as pd
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, FOAF, RDFS, XSD #import already in RDFlib integrated namespaces
from rdflib import URIRef, BNode, Literal         #in case we need URIs, blank nodes, or literals
from pathlib import Path


In [64]:
DATA = Path("test_data")

with open(DATA / "test_turtle.ttl", encoding="utf-8") as f:
    data = f.read()
    # print(data)


In [65]:
g=Graph()
g.parse(data=data,format="turtle")
print(g)


[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].


In [66]:
print(len(g))
for s, p, o in list(g)[:5]:
    print(s, p, o)


49
http://www.w3.org/kg2023/BallpointPen http://purl.org/dc/terms/date 1938
http://www.w3.org/kg2023/BallpointPen http://dbpedia.org/ontology/country http://www.w3.org/kg2023/Hungary
http://www.w3.org/kg2023/Walkman http://www.w3.org/2000/01/rdf-schema#label Walkman
http://www.w3.org/kg2023/WorldWideWeb http://purl.org/dc/terms/date 1989
http://www.w3.org/kg2023/Penicillin http://dbpedia.org/ontology/country http://www.w3.org/kg2023/UnitedKingdom


In [80]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dc: <http://purl.org/dc/terms/>

SELECT ?inventionName ?invention ?date
WHERE
{
  ?invention rdf:type dbo:Invention .
  ?invention dc:date ?date .
  ?invention rdfs:label ?inventionName .
  Filter (xsd:integer(?date) > 1974) .
  Filter (xsd:integer(?date) <= 1979) .
}
"""


query_result = g.query(query)

print(query_result.bindings)



[{rdflib.term.Variable('invention'): rdflib.term.URIRef('http://www.w3.org/kg2023/Walkman'), rdflib.term.Variable('date'): rdflib.term.Literal('1979'), rdflib.term.Variable('inventionName'): rdflib.term.Literal('Walkman', lang='en')}, {rdflib.term.Variable('invention'): rdflib.term.URIRef('http://www.w3.org/kg2023/Walkman'), rdflib.term.Variable('date'): rdflib.term.Literal('1979'), rdflib.term.Variable('inventionName'): rdflib.term.Literal('Walkman', lang='de')}]


In [81]:
df = pd.DataFrame(query_result, columns=["Invention Name", "invention", "date"])
df

,Invention Name,invention,date
0,Walkman,http://www.w3.org/kg2023/Walkman,1979
1,Walkman,http://www.w3.org/kg2023/Walkman,1979


In [ ]:
import mkwikidata
import requests

def run_wikidata_query(query, params={}):
    # Wikidata now requires a descriptive User-Agent header, otherwise it
    # returns HTTP 403 with a plain-text body (which breaks mkwikidata's r.json()).
    query = mkwikidata.Template(query).substitute(**params)
    headers = {"User-Agent": "KG-Course-Notebook/1.0 (arne.martensen@protonmail.com)"}
    r = requests.post(
        "https://query.wikidata.org/sparql",
        params={"format": "json", "query": query},
        headers=headers,
    )
    r.raise_for_status()
    return r.json()


In [ ]:
query = """
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>

SELECT ?movie (COUNT(?actors) AS ?num_actors) 
WHERE {     
  ?movie wdt:P31 wd:Q11424 .                      
  ?movie wdt:P57  wd:Q271967 .      
  ?movie wdt:P161 ?actors .            
  } GROUP BY (?movie) HAVING (COUNT(?actors)>5)
"""


In [ ]:
query_result = run_wikidata_query(query, params={})
results_df = pd.json_normalize(query_result["results"]["bindings"])
results_df[["movie.value", "num_actors.value"]]